# Early Signals: a walkthrough of the whole pipeline

This notebook runs the project **step by step** and explains what each piece of code does and why.
It calls the same functions the command line uses (`python -m early_signals run ...`), so what you
learn here is exactly what produced the paper's numbers.

**How to use it**
- Run cells top to bottom (Shift+Enter). The first run uses **synthetic** data (made-up people), so it is safe
  to run anywhere and to share.
- To run on your real NHATS files, change `SOURCE` in the next cell to `"nhats"`.

> **NHATS rule:** with real data, cell outputs can contain information about individuals. The notebook only
> prints aggregates (counts, rates, metrics) when `SOURCE == "nhats"`, but still **do not share or upload a
> notebook that has been run on NHATS**. Before sharing, use *Clear All Outputs*.

**Where the code lives** (all under `src/early_signals/`):

| Step | File | What it does |
|---|---|---|
| 1 | `synth.py`, `nhats_loader.py`, `nhats_extra.py`, `dementia.py` | Load data into one table: one row per person per year |
| 2 | `config/variables.yaml` | Which NHATS variable means what, and how codes are recoded |
| 3 | `features.py` | Build prediction windows (t-1 → t → t+1) and features |
| 4 | `evaluate.py` | Split train/test, compute metrics, fairness audit |
| 5 | `models.py`, `tuning.py` | Fit models; tune the expanded model on training data only |
| 6 | `explain.py` | “What if this change had not happened?” explanations |
| 7 | `app_model.py` | The small model the Steady app runs |
| — | `pipeline.py`, `report.py`, `paper.py`, `__main__.py` | Glue: run everything and write reports |

In [ ]:
# ---- settings -------------------------------------------------------------
SOURCE = "synthetic"          # "synthetic" (safe, made-up people) or "nhats" (your real files)
NHATS_DIR = "~/Downloads/NHATS Data"
SCALE = 0.5                   # synthetic only: size of the fake sample (1.0 = 12,000 people)

import os, sys, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "src")))   # run from notebooks/
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "src")))         # or from the repo root
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 160)
SHOW_ROWS = SOURCE == "synthetic"     # only ever print individual rows for synthetic data
print("Using", SOURCE, "data")

## 1. Load the data into one tidy table

Everything downstream works on a **long panel**: one row per person per year (NHATS “round”), with the same
column names whether the data are synthetic or real. That is the key design choice: the loader is the only
part that knows about NHATS variable names.

- `synth.generate()` makes fake people whose abilities slowly decline, including deaths and moves into care.
- `nhats_loader.load_panel()` reads each round's Stata file, keeps only the columns we need, and recodes them
  using `config/variables.yaml`, `nhats_extra.py` (household tasks, physical tests, ...) and `dementia.py`
  (the official NHATS dementia classification, re-written in Python).

In [ ]:
from early_signals.pipeline import load
panel = load(SOURCE, os.path.expanduser(NHATS_DIR), scale=SCALE)
print(f"{panel.person_id.nunique():,} people, {len(panel):,} person-years, {panel.shape[1]} columns")
print("\nWhere people were each year (counts):")
print(panel.pivot_table(index="round", columns="status", values="person_id", aggfunc="count").fillna(0).astype(int))
if SHOW_ROWS:
    display(panel[["person_id", "round", "status", "age", "bathing_diff", "bathing_device", "go_out_freq", "bathing_help"]].head(8))

### What the columns mean

The column names are defined once in `schema.py`. For each of the 7 activities (going outside, moving around
inside, getting out of bed, eating, bathing, toileting, dressing) there are up to five columns:

| Suffix | Meaning | NHATS source |
|---|---|---|
| `_help` | got help from another person (this is what we predict) | derived help measures |
| `_diff` | difficulty doing it by themselves | derived self-difficulty measures |
| `_device` | uses a device (cane, grab bar, shower seat...) | derived device measures |
| `_less` | does it less often than a year ago | “compared to a year ago” items |
| `_wout` | went without it (no help / too hard) | “went without” items |

In [ ]:
from early_signals import schema
print("Activities:", schema.ACTIVITIES)
print("\nItems tracked over time in the original (v1) model:", len(schema.V1_ITEMS))
print("Extra items in the expanded model:", len(schema.EXTRA_ITEMS))
print("\nA few plain-language labels:")
for k in ["bathing_diff", "bathing_device", "go_outside_wout", "go_out_freq", "walk_time", "iadl_shopping_hhelp"]:
    print(f"  {k:22s} {schema.ITEM_LABELS.get(k)}")

## 2. The variable map (NHATS → our names)

`config/variables.yaml` is where the NHATS codebook knowledge lives. Each entry names the NHATS variable (with
`{r}` standing for the round number) and a **recode rule**. NHATS stores “missing” as negative numbers
(-1 inapplicable, -7 refused, -8 don't know, -9 missing), which the recode turns into `NaN` unless a rule says
otherwise.

In [ ]:
from early_signals.nhats_loader import load_config, _recode
cfg = load_config()
for k in ["bathing_help", "bathing_diff", "bathing_device", "go_out_freq", "residence"]:
    spec = cfg["items"][k]
    print(f"{k:16s} <- {spec['nhats'].format(r=5):16s} recode: {cfg['recodes'][spec['recode']]}")

# try a recode by hand: NHATS derived help codes 1 = no, 2 = yes, 8 = not done, -1 inapplicable
print("\nExample recode of [1, 2, 8, -1, -9] with the help rule:",
      _recode(pd.Series([1, 2, 8, -1, -9]), cfg["recodes"]["help_d"]).tolist())

## 3. Prediction windows: the heart of the study

Each training example is a **window** anchored at year *t* for one person:

```
year t-1  →  year t              →  year t+1
history      status + change         outcome: newly getting help?
```

`features.build_windows()` lines up each person's rows for t-1, t and t+1 and keeps a window only if the person:
1. lived in the community (not a nursing home) at t-1 and t, and
2. got **no help** with any of the 7 activities at t (they are “at risk” of newly needing help).

The outcome `y_any` = 1 if they got help with at least one activity at t+1. Nothing from t+1 is ever used as
an input; that is what makes it a real prediction.

In [ ]:
from early_signals.features import build_windows
windows = build_windows(panel)
analytic = windows[windows.y_any.notna()].copy()
analytic["y_any"] = analytic["y_any"].astype(int)
print(f"{len(windows):,} at-risk windows; {len(analytic):,} with the outcome observed at home at t+1")
print(f"Overall rate of NEW help a year later: {analytic.y_any.mean():.1%}")
rate = analytic.groupby("round").y_any.mean()
ax = (rate * 100).plot(marker="o", figsize=(6, 3), title="New help within a year, by anchor year t")
ax.set_ylabel("%"); plt.show()
print("\nWhat happened at t+1 (counts):"); print(windows.exit_next.value_counts())

Each window stores every item twice: `item__prev` (year t-1) and `item__cur` (year t). That is all the
information the features need.

In [ ]:
cols = [c for c in analytic.columns if c.startswith("bathing_")]
print(cols)
if SHOW_ROWS:
    display(analytic[["person_id", "round"] + ["bathing_diff__prev", "bathing_diff__cur", "bathing_device__prev",
                                               "bathing_device__cur", "y_any"]].head(6))

## 4. Features: “status” vs “status + change”

`features.make_X(windows, feature_set)` turns windows into the numeric table a model sees.

- **`status`**: only the values at year t (a snapshot), plus age, sex, living alone, proxy, and pandemic flags.
- **`change`**: the snapshot **plus** how each item changed since t-1:
  `d_<item>` = difference for ordered items, `new_<item>` = 1 if something newly appeared (e.g. a new device),
  plus counts like `n_new_devices`.
- `change_plus` and `change_plus_2yr` add the expanded measures and two-year trends (Section 8).

Comparing `status` with `change` is the paper's main question.

In [ ]:
from early_signals.features import make_X
for fs in ["status", "change", "change_plus", "change_plus_2yr"]:
    X = make_X(analytic, fs)
    print(f"{fs:16s} {X.shape[1]:4d} features, e.g. {[c for c in X.columns if not c.startswith('cur_')][:5]}")

Xc = make_X(analytic, "change")
print("\nHow often each kind of change happens (share of windows):")
print(Xc[["new_bathing_device", "new_go_outside_wout", "n_new_devices"]].gt(0).mean().round(3))

## 5. Train/test split that can't cheat

`evaluate.temporal_person_split()` does two things at once:
- **Person-disjoint:** 30% of people are held out entirely, so the model is tested on strangers.
- **Time-ordered:** training uses outcomes up to a split year; testing uses later years only.

A random row-level split would put the same person in both sets and make results look better than they are.

In [ ]:
from early_signals.evaluate import temporal_person_split
split_round = int(analytic["round"].max()) - 3
train, test = temporal_person_split(analytic, split_round, seed=0)
print(f"train {len(train):,} windows (outcome year ≤ round {split_round}); test {len(test):,} windows (t ≥ {split_round})")
print("people in both sets:", len(set(train.person_id) & set(test.person_id)))

## 6. The four pre-specified models

`models.model_specs()` returns the models compared in the paper. Each entry is (feature set, model factory).
The logistic models impute missing values with the median and add “was missing” indicator columns; tree models
handle missing values themselves.

In [ ]:
from early_signals.models import model_specs
from sklearn.metrics import roc_auc_score
preds, fitted = {}, {}
for name, (fset, factory) in model_specs().items():
    m = factory().fit(make_X(train, fset), train.y_any.values)
    preds[name] = m.predict_proba(make_X(test, fset))[:, 1]
    fitted[name] = (m, fset)
    print(f"{name:32s} features={fset:7s} test AUROC {roc_auc_score(test.y_any, preds[name]):.3f}")

**AUROC** = the chance the model ranks a random person who went on to need help above a random person who
didn't (0.5 = coin flip, 1.0 = perfect).

## 7. Evaluation beyond AUROC

`evaluate.py` also checks whether predicted percentages are *right* (calibration) and whether acting on them
helps (decision curves). The paper's confidence intervals come from `cluster_bootstrap`, which resamples
**people** (not rows), because one person contributes several windows.

In [ ]:
from early_signals import evaluate as ev
names = list(preds); base, primary = names[0], names[-1]
y = test.y_any.values
for k in (base, primary):
    m = ev.core_metrics(y, preds[k])
    print(f"{k:32s} AUROC {m['auroc']:.3f}  AUPRC {m['auprc']:.3f}  observed/expected {m['obs_exp']:.2f}  slope {m['cal_slope']:.2f}")

boot = ev.cluster_bootstrap(test, {base: preds[base], primary: preds[primary]}, "y_any", B=100)
g = boot.set_index("model").loc[primary]
print(f"\nGain from change features: {g.auroc_gain_vs_baseline:+.3f} (95% CI {g.gain_lo:+.3f} to {g.gain_hi:+.3f})")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for k in (base, primary):
    c = ev.calibration_bins(y, preds[k]); axes[0].plot(c.mean_pred, c.obs_rate, "o-", label=k)
    d = ev.decision_curve(y, preds[k]); axes[1].plot(d.threshold, d.net_benefit, label=k)
mx = axes[0].get_xlim()[1]; axes[0].plot([0, mx], [0, mx], "--", color="grey")
axes[0].set(title="Calibration: predicted vs observed", xlabel="predicted", ylabel="observed")
axes[1].plot(d.threshold, d.treat_all, "--", color="grey", label="review everyone"); axes[1].axhline(0, color="lightgrey")
axes[1].set(title="Decision curve", xlabel="risk threshold", ylabel="net benefit", xlim=(0.02, 0.4), ylim=(-0.01, None))
for a in axes: a.legend(fontsize=7)
plt.show()

> **Don't be surprised by synthetic results.** The fake data were generated so that a person's *current* state
> already carries most of the signal, so change features add little there. On the real NHATS data they added
> **+0.029 AUROC (95% CI 0.014–0.044)**, the paper's main result. Switch `SOURCE` to `"nhats"` to see it.

### Fairness audit

`subgroup_audit` repeats the metrics inside each group. **O/E > 1** means the model *under*-estimates risk for
that group, the pattern the paper found for Black, Hispanic and Medicaid-enrolled participants.

In [ ]:
thr = float(np.quantile(fitted[primary][0].predict_proba(make_X(train, fitted[primary][1]))[:, 1], 0.8))
fair = ev.subgroup_audit(test, preds[primary], "y_any", thr, schema.SUBGROUP_COLS, B=50)
print(fair[["subgroup", "level", "n", "prevalence", "auroc", "obs_exp", "sensitivity", "alert_rate"]].round(3).to_string(index=False))

## 8. Explanations: “what if this change had not happened?”

`explain.change_contributions()` re-scores each person as if one area of change (e.g. bathing) had stayed the
same as last year, and records how much the risk estimate drops. `global_change_importance()` summarizes that
separately for people who got **worse** and who got **better**. Mixing the two would let improvements cancel
out declines.

In [ ]:
from early_signals.explain import change_contributions, global_change_importance
model, fset = fitted[primary]
t0 = test.reset_index(drop=True)
contrib = change_contributions(model, fset, t0)
imp = global_change_importance(contrib, t0.y_any.values)
print(imp[imp.direction != "better"][["group", "direction", "n", "mean_contribution", "observed_new_help"]]
      .assign(mean_contribution=lambda d: (d.mean_contribution * 100).round(1),
              observed_new_help=lambda d: (d.observed_new_help * 100).round(1)).to_string(index=False))

## 9. The expanded model (secondary analysis)

`tuning.select_and_fit()` tries three feature sets × {logistic, boosted trees, their average} using
**5-fold cross-validation on the training data only**, then picks the best and recalibrates it. The test set is
touched once, afterwards. Here we use very few hyperparameter configurations so it runs in a minute or two;
the real run uses 12 per feature set.

In [ ]:
from early_signals.tuning import select_and_fit
exp_model, chosen, cv = select_and_fit(train, "y_any", n_trees_configs=2, seed=0)
p_exp = exp_model.predict_proba(make_X(test, exp_model.fset))[:, 1]
print(f"\nExpanded model on the test set: AUROC {roc_auc_score(y, p_exp):.3f} (v1 primary: {roc_auc_score(y, preds[primary]):.3f})")
print(cv.groupby(["feature_set", "family"]).cv_auroc.max().round(3).unstack())

## 10. The model inside the Steady app

The app can only ask questions a person can answer at home, so `app_model.py` maps NHATS items onto the app's
check-in questions (e.g. “how did bathing go?” → easy / used something / hard / went without), trains a small
logistic model on those, and exports its coefficients. On your Mac:
`python -m early_signals export-app-model --nhats-dir ... --out app_model.json`.

In [ ]:
from early_signals.app_model import app_features, fit_app_model
print("App features:", list(app_features(test.head(3)).columns))
am = fit_app_model(train, test, "y_any")
top = sorted(am["features"], key=lambda f: -abs(f["coef"]))[:8]
print("\nLargest coefficients (per 1 standard deviation):")
for f in top: print(f"  {f['label']:32s} {f['name']:14s} {f['coef']:+.2f}")

## 11. Running everything at once

Everything above is what `pipeline.run()` does in one go, plus writing `report.md`, `report_expanded.md`,
`paper_tables.md`, the figures, and CSVs. From a terminal:

```bash
python -m early_signals run --source nhats --nhats-dir ~/Downloads/"NHATS Data" --out results_final
```

**Suggested reading order in VS Code:** `schema.py` → `features.py` → `evaluate.py` → `models.py` →
`pipeline.py` → `explain.py` → `tuning.py` → `nhats_loader.py` / `nhats_extra.py` / `dementia.py` → `app_model.py`.
The tests in `tests/test_pipeline.py` are short and show the guarantees the code must keep (for example, that no
future information leaks into the features).